# AI Air Quality Detection — Nine Pakistani Cities

This notebook follows the teacher's project requirements in a simple order:
1. Explain the problem.
2. Describe the dataset.
3. Clean and prepare the data.
4. Study the data using tables and graphs (EDA).
5. Train three machine-learning models.
6. Compare the models and choose the best one.
7. Save the selected model for the Streamlit app.

**Important:** The target contains five project-defined AQI classes (1–5). We retain only records whose AQI is already a whole number from 1 to 5. We do not create classes by clipping or rounding other AQI values. These are project classes, not a claim that they are official AQI categories.

## 1. Problem Statement

Air pollution is difficult to understand from pollutant numbers alone. Our project uses historical air-quality data from nine Pakistani cities.

The machine-learning model uses eight pollutant measurements and the city name to predict an air-quality class from 1 to 5.

The Streamlit app has two options:
- Choose a city and use its latest available historical record.
- Enter pollutant values manually.

**Note:** The app uses historical data. It is not a live air-quality sensor.

## 2. Dataset Description

We use nine CSV files, one for each city. Each file contains a timestamp, AQI, and eight pollutant measurements.

| Column | Meaning |
|---|---|
| CO | Carbon monoxide |
| NO | Nitric oxide |
| NO2 | Nitrogen dioxide |
| O3 | Ozone |
| SO2 | Sulfur dioxide |
| PM2.5 | Fine particles in the air |
| PM10 | Larger particles in the air |
| NH3 | Ammonia |
| AQI | Value that we convert into our prediction class |
| City | City name added by our code |

Dataset files are hosted here: [Pakistan Air Quality Dataset — Hugging Face](https://huggingface.co/datasets/mk12rule/pakistan_air_quality_dataset).

The page contains the matching city files, but does not explain the original data collector. We therefore describe it as the page hosting the files, not as proof of the original data provider.

In [ ]:
# Step 1: Import the libraries we need.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Find the project folder. The dataset folder should be inside it.
BASE_DIR = os.getcwd()
if not os.path.isdir(os.path.join(BASE_DIR, "dataset")):
    BASE_DIR = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(BASE_DIR, "dataset")

# These are the eight input columns and the column we want to predict.
FEATURES = ["CO", "NO", "NO2", "O3", "SO2", "PM2.5", "PM10", "NH3"]
TARGET = "AQI"

# Each city is connected to its CSV file.
CITY_FILES = {
    "Faisalabad": "historical_air_pollution_all_faisalabad.csv",
    "Islamabad": "historical_air_pollution_all_islamabad.csv",
    "Karachi": "historical_air_pollution_all_karachi.csv",
    "Lahore": "historical_air_pollution_all_lahore.csv",
    "Larkana": "historical_air_pollution_all_larkana.csv",
    "Multan": "historical_air_pollution_all_multan.csv",
    "Peshawar": "historical_air_pollution_all_peshawar.csv",
    "Quetta": "historical_air_pollution_all_quetta.csv",
    "Rawalpindi": "historical_air_pollution_all_rawalpindi.csv"
}

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError("The dataset folder was not found. Keep it beside the notebook.")

print("Project folder:", BASE_DIR)
print("Dataset folder:", DATA_DIR)
print("Number of city files expected:", len(CITY_FILES))

In [ ]:
# Step 2: Read the nine city files and combine them into one table.
all_cities = []
total_invalid_aqi = 0

for city in CITY_FILES:
    file_name = CITY_FILES[city]
    file_path = os.path.join(DATA_DIR, file_name)

    city_data = pd.read_csv(file_path)

    # Keep only the columns needed in this project.
    city_data = city_data[["Timestamp", TARGET] + FEATURES].copy()

    # Convert dates and numbers into the correct data types.
    city_data["Timestamp"] = pd.to_datetime(city_data["Timestamp"], errors="coerce")
    for column in [TARGET] + FEATURES:
        city_data[column] = pd.to_numeric(city_data[column], errors="coerce")

    # In these files, -9999 means that a measurement is missing or invalid.
    city_data = city_data.replace(-9999, np.nan)

    # Remove duplicate rows and rows without a date or AQI value.
    city_data = city_data.drop_duplicates()
    city_data = city_data.dropna(subset=["Timestamp", TARGET])

    # Keep only whole-number AQI values from 1 to 5.
    # The small difference check handles decimal storage by the computer.
    aqi_values = city_data[TARGET]
    whole_number_check = (aqi_values - aqi_values.round()).abs() <= 0.000001
    valid_aqi = aqi_values.between(1, 5) & whole_number_check

    total_invalid_aqi = total_invalid_aqi + (len(city_data) - valid_aqi.sum())
    city_data = city_data[valid_aqi].copy()
    city_data[TARGET] = city_data[TARGET].round().astype(int)

    # Add the city name so the model can use it as an input.
    city_data["City"] = city
    all_cities.append(city_data)

# Join all nine city tables into one table.
data = pd.concat(all_cities, ignore_index=True)

# Keep missing pollutant values for now.
# They will be filled by the model imputer using training data only.
data = data.sort_values(["City", "Timestamp"])
data = data.reset_index(drop=True)

print("Total rows and columns:", data.shape)
print("Invalid AQI records excluded:", total_invalid_aqi)
print("Missing values in each column:")
display(data.isnull().sum())
print("Duplicate rows:", data.duplicated().sum())
print("AQI classes:", sorted(data[TARGET].unique()))
display(data.head())


## 3. Data Preprocessing

Before training, we prepare the data:

1. Read and combine the nine city files.
2. Convert dates and pollutant columns into the correct data types.
3. Treat `-9999` as a missing or invalid value.
4. Remove duplicate rows and rows without a valid date or AQI value.
5. Retain only AQI values that are already whole numbers from 1 to 5. Values outside this range or non-whole-number values are excluded, not clipped or rounded into a class.
6. Keep missing pollutant values until model preprocessing, where median imputation is fitted using training data only.
7. Add the city name as a categorical input feature.
8. Apply one-hot encoding to the city feature and standard scaling for Logistic Regression.

**In simple words:** preprocessing means cleaning and preparing the data before giving it to the machine-learning models.

In [ ]:
# Step 3: Explore the data using tables and graphs.

# Count how many records belong to each city.
city_counts = data["City"].value_counts()
print("Records for each city:")
display(city_counts)

# Show basic information such as average, minimum, maximum and median.
print("Summary of pollutant values:")
display(data[FEATURES + [TARGET]].describe())

# Count the number of records in each AQI class.
class_counts = data[TARGET].value_counts().sort_index()
print("Number of records in each AQI class:")
display(class_counts)

# Draw a bar chart for AQI classes.
plt.figure(figsize=(8, 4))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Number of Records in Each AQI Class")
plt.xlabel("AQI class")
plt.ylabel("Number of records")
plt.show()

# Draw a bar chart for the cities.
plt.figure(figsize=(10, 4))
plt.bar(city_counts.index, city_counts.values)
plt.title("Number of Records in Each City")
plt.xlabel("City")
plt.ylabel("Number of records")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Step 4: Check how pollutant values are related to AQI.

# Correlation tells us whether two values tend to move together.
correlation = data[FEATURES + [TARGET]].corr()

print("Correlation values:")
display(correlation)

# Show the correlation values as a heatmap.
plt.figure(figsize=(9, 7))
plt.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=45)
plt.yticks(range(len(correlation.index)), correlation.index)
plt.title("Correlation Between Pollutants and AQI")
plt.show()

# Draw a scatter plot to compare PM2.5 with AQI.
plt.figure(figsize=(8, 5))
plt.scatter(data["PM2.5"], data[TARGET], alpha=0.2)
plt.title("PM2.5 Measurements and AQI Class")
plt.xlabel("PM2.5")
plt.ylabel("AQI class")
plt.show()

# Show AQI class percentages.
print("Percentage of records in each AQI class:")
class_counts = data[TARGET].value_counts().sort_index()
print(class_counts / len(data) * 100)

# Show which pollutant has the strongest correlation with AQI.
print("Correlation of each pollutant with AQI:")
print(correlation[TARGET].drop(TARGET).sort_values(ascending=False))

In [ ]:
# Step 5: Print simple findings from our data.

most_common_class = class_counts.idxmax()
least_common_class = class_counts.idxmin()

print("AUTOMATIC EDA FINDINGS")
print("Most common AQI class:", most_common_class)
print("Least common AQI class:", least_common_class)
print("City with most records:", city_counts.idxmax(), city_counts.max())
print("City with fewest records:", city_counts.idxmin(), city_counts.min())
print("Median PM2.5:", data["PM2.5"].median())

# Find the pollutant most strongly related to AQI.
correlation_values = correlation[TARGET].drop(TARGET)
strongest_pollutant = correlation_values.abs().idxmax()
print("Pollutant most strongly correlated with AQI:", strongest_pollutant)
print("Correlation value:", correlation_values[strongest_pollutant])

# Show the median PM2.5 value in each AQI class.
print("Median PM2.5 in each AQI class:")
for class_number in [1, 2, 3, 4, 5]:
    class_data = data[data[TARGET] == class_number]
    if len(class_data) > 0:
        print("Class", class_number, ":", class_data["PM2.5"].median())

### EDA Interpretation

EDA means **Exploratory Data Analysis**. It means looking at our data using tables, numbers, and graphs before training models.

In this project, we check:
- how many records are in each AQI class;
- how many records come from each city;
- the typical pollutant values;
- how pollutant values are related to AQI; and
- how PM2.5 changes across the five project classes.

The code above calculates these findings from the dataset. The numbers can change if the dataset changes.

### Simple Explanation of Our EDA Results

The graphs, tables, and automatic findings above are calculated from the cleaned dataset each time the notebook is run. Use those current outputs when describing the results.

In particular, check:
1. Which AQI class has the most and fewest records.
2. Which city has the most and fewest records.
3. The median PM2.5 value.
4. Which pollutant has the strongest correlation with AQI.
5. How median PM2.5 changes across the five project-defined AQI classes.

**Report summary:** Describe the patterns shown by the latest notebook outputs. These are relationships in this dataset and do not prove cause and effect.

## 4. Split the Data into Training and Testing Sets

We divide the data into two parts:
- **80% training data:** the models learn from this part.
- **20% testing data:** we check how well the models work on data they did not train on.

We keep the records in date order for each city. This means older records go into training and newer records go into testing.

In [ ]:
# Step 6: Split each city's records by date.
train_data = []
test_data = []

for city in CITY_FILES:
    city_data = data[data["City"] == city]
    city_data = city_data.sort_values("Timestamp")

    # First 80% for training and last 20% for testing.
    split_point = int(len(city_data) * 0.80)
    train_data.append(city_data.head(split_point))
    test_data.append(city_data.tail(len(city_data) - split_point))

# Join the training parts and testing parts.
train_df = pd.concat(train_data)
test_df = pd.concat(test_data)

# X means input columns. y means the answer we want to predict.
X_train = train_df[FEATURES + ["City"]]
y_train = train_df[TARGET]
X_test = test_df[FEATURES + ["City"]]
y_test = test_df[TARGET]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training class counts:")
display(y_train.value_counts().sort_index())
print("Testing class counts:")
display(y_test.value_counts().sort_index())


## 5. Train Three Machine-Learning Models

We train three classification models because the output is one of five classes:

1. **Logistic Regression:** uses mathematical relationships between the inputs to choose a class.
2. **Decision Tree:** makes decisions by asking questions about the input values, like a flowchart.
3. **Random Forest:** combines many decision trees and uses their combined result.

We train all three models on the same training data and test them on the same testing data. This makes the comparison fair.

**Library note:** We use ready-made Scikit-learn tools for missing values, city encoding, scaling, and model training. We do not write these algorithms from scratch. The preprocessing tools are included with each model so the app can use the same steps when making predictions.


In [ ]:
# Step 7: Prepare the data and train the three models.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# The computer needs city names to be changed into numbers.
# OneHotEncoder does this for us.
# SimpleImputer fills any missing number using the median.

# Logistic Regression: fill missing values, scale numbers, and encode city.
logistic_preparation = ColumnTransformer([
    ("numbers", Pipeline([
        ("fill_missing", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), FEATURES),
    ("city", OneHotEncoder(handle_unknown="ignore"), ["City"])
])

# Decision Tree: fill missing values and encode city. Scaling is not needed.
tree_preparation = ColumnTransformer([
    ("numbers", SimpleImputer(strategy="median"), FEATURES),
    ("city", OneHotEncoder(handle_unknown="ignore"), ["City"])
])

# Random Forest uses the same simple preparation as Decision Tree.
forest_preparation = ColumnTransformer([
    ("numbers", SimpleImputer(strategy="median"), FEATURES),
    ("city", OneHotEncoder(handle_unknown="ignore"), ["City"])
])

# Connect each preparation step with its machine-learning model.
models = {
    "Logistic Regression": Pipeline([
        ("prepare_data", logistic_preparation),
        ("model", LogisticRegression(max_iter=500, class_weight="balanced"))
    ]),
    "Decision Tree": Pipeline([
        ("prepare_data", tree_preparation),
        ("model", DecisionTreeClassifier(max_depth=14, min_samples_leaf=8, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("prepare_data", forest_preparation),
        ("model", RandomForestClassifier(n_estimators=100, max_depth=16, min_samples_leaf=8, class_weight="balanced_subsample", n_jobs=-1, random_state=42))
    ])
}

# Train and test each model.
trained_models = {}
predictions = {}
results = []

for model_name in models:
    print("Training:", model_name)
    current_model = models[model_name]
    current_model.fit(X_train, y_train)

    predicted_values = current_model.predict(X_test)
    trained_models[model_name] = current_model
    predictions[model_name] = predicted_values

    # Calculate scores to measure model performance.
    accuracy = accuracy_score(y_test, predicted_values)
    precision = precision_score(y_test, predicted_values, average="macro", zero_division=0)
    recall = recall_score(y_test, predicted_values, average="macro", zero_division=0)
    f1 = f1_score(y_test, predicted_values, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_test, predicted_values, average="weighted", zero_division=0)

    results.append([model_name, accuracy, precision, recall, f1, weighted_f1])

    print("Results for", model_name)
    print(classification_report(y_test, predicted_values, zero_division=0))

## 6. Model Evaluation and Comparison

We use scores to check how well each model predicts:

- **Accuracy:** how many predictions were correct overall.
- **Precision:** when the model predicts a class, how often it is correct.
- **Recall:** how many actual examples of a class the model finds.
- **F1-score:** combines precision and recall into one score.
- **Confusion matrix:** a table that shows correct and incorrect predictions for each class.

We compare the three models using the same testing data. We choose the model with the highest macro F1 score because each AQI class gets equal importance in this score.


In [ ]:
# Step 8: Compare the results of the three models.
from sklearn.metrics import confusion_matrix

results_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)", "F1 (weighted)"
])

# Put the model with the highest macro F1 score at the top.
results_df = results_df.sort_values(by="F1 (macro)", ascending=False)
print("Model comparison:")
display(results_df)

# Show a confusion-matrix table for each model.
for model_name in predictions:
    print("Confusion matrix for:", model_name)
    matrix = confusion_matrix(y_test, predictions[model_name], labels=[1, 2, 3, 4, 5])
    matrix_table = pd.DataFrame(
        matrix,
        index=["Actual 1", "Actual 2", "Actual 3", "Actual 4", "Actual 5"],
        columns=["Predicted 1", "Predicted 2", "Predicted 3", "Predicted 4", "Predicted 5"]
    )
    display(matrix_table)

# Select the first model because the table is sorted by macro F1 score.
best_name = results_df["Model"].values[0]
best_model = trained_models[best_name]
print("Best model:", best_name)
print("Best macro F1 score:", results_df["F1 (macro)"].values[0])


In [ ]:
# Step 9: Save the comparison table and the selected model for the Streamlit app.
import joblib

MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

# Save the model scores in a CSV file.
results_df.to_csv(os.path.join(MODEL_DIR, "model_comparison.csv"), index=False)

# Save the selected model so the app can load it later.
joblib.dump(best_model, os.path.join(MODEL_DIR, "best_air_quality_classifier.joblib"))

print("Model comparison and best model have been saved in:", MODEL_DIR)

## 7. Model Understanding and Application Integration

The notebook compares three models and selects the one with the highest macro F1 score.

- Logistic Regression is our simple mathematical model.
- Decision Tree makes decisions using question-like rules.
- Random Forest combines many decision trees.

The selected model is saved in the `models` folder. The Streamlit app loads this saved model to make predictions from the pollutant values and city name.

### Important points to explain in the presentation

- We used historical data from nine Pakistani cities.
- The eight pollutant measurements and city name are the inputs.
- The output is one of our five project-defined classes.
- We cleaned missing and duplicate data before training.
- We trained three models and compared their scores.
- The app uses historical records, not live sensor readings.
- Our classes 1–5 are project-defined and should not be described as official AQI categories unless we verify them against an official standard.